In [1]:
import onnx

In [2]:
model = onnx.load("model.onnx")

## Inspect model graph

In [4]:
print(onnx.helper.printable_graph(model.graph))

graph main_graph (
  %obs[FLOAT, batch_sizex12]
  %state_ins[FLOAT, batch_size]
) initializers (
  %mlp_extractor.policy_net.0.weight[FLOAT, 64x12]
  %mlp_extractor.policy_net.0.bias[FLOAT, 64]
  %mlp_extractor.policy_net.2.bias[FLOAT, 64]
  %action_net.weight[FLOAT, 2x64]
  %action_net.bias[FLOAT, 2]
  %mlp_extractor.policy_net.2.weight[FLOAT, 64x64]
) {
  %linear = Gemm[alpha = 1, beta = 1, transA = 0, transB = 1](%obs, %mlp_extractor.policy_net.0.weight, %mlp_extractor.policy_net.0.bias)
  %tanh = Tanh(%linear)
  %linear_1 = Gemm[alpha = 1, beta = 1, transA = 0, transB = 1](%tanh, %mlp_extractor.policy_net.2.weight, %mlp_extractor.policy_net.2.bias)
  %tanh_1 = Tanh(%linear_1)
  %output = Gemm[alpha = 1, beta = 1, transA = 0, transB = 1](%tanh_1, %action_net.weight, %action_net.bias)
  %state_outs = Identity(%state_ins)
  return %output, %state_outs
}


C:\Users\emanu\AppData\Local\Temp\ipykernel_12024\2222816960.py:1: DeprecationWarning: Deprecated since 1.19. Consider using onnx.printer.to_text() instead.
  print(onnx.helper.printable_graph(model.graph))


In [10]:
from onnx import numpy_helper

params = {}
for tensor in model.graph.initializer:
    name = tensor.name
    array = numpy_helper.to_array(tensor)
    params[name] = array
    
    print(f"Name: {name}")
    print(f"Shape: {array.shape}")
    print("-" * 40)

Name: mlp_extractor.policy_net.0.weight
Shape: (64, 12)
----------------------------------------
Name: mlp_extractor.policy_net.0.bias
Shape: (64,)
----------------------------------------
Name: mlp_extractor.policy_net.2.bias
Shape: (64,)
----------------------------------------
Name: action_net.weight
Shape: (2, 64)
----------------------------------------
Name: action_net.bias
Shape: (2,)
----------------------------------------
Name: mlp_extractor.policy_net.2.weight
Shape: (64, 64)
----------------------------------------


## Print to json

In [11]:
import json

params_json = {}

for name, array in params.items():
    params_json[name] = array.tolist()

with open("weights.json", "w") as f:
    json.dump(params_json, f, indent=2)

## Replicate model

In [16]:
import numpy as np
def forward(obs, params):
    W0, b0 = params["mlp_extractor.policy_net.0.weight"], params["mlp_extractor.policy_net.0.bias"]
    W1, b1 = params["mlp_extractor.policy_net.2.weight"], params["mlp_extractor.policy_net.2.bias"]
    W2, b2 = params["action_net.weight"], params["action_net.bias"]

    Z1 = obs @ W0.T + b0
    A1 = np.tanh(Z1)

    Z2 = A1 @ W1.T + b1
    A2 = np.tanh(Z2)

    Z3 = A2 @ W2.T + b2

    return Z3

In [13]:
inp=np.random.random(size=12)
inp

array([0.92936797, 0.88937824, 0.40962972, 0.36246534, 0.02878767,
       0.43262709, 0.56968231, 0.87754235, 0.66167488, 0.91388427,
       0.7021675 , 0.13770642])

In [17]:
forward(inp,params)

array([ 0.14832648, -1.31082861])